# 04. Pydantic による型安全な構造化出力（response_schema）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/ja/04_structured_outputs.ipynb)

プロンプトだけで「JSON 形式のみで回答してください」と指示する手法は、前置き文の混入や ` ```json ` コードフェンスの付与によって本番環境で頻繁にパースエラーを引き起こします。`response_mime_type="application/json"` と `response_schema=<Pydantic BaseModel>` を併用すると **文法制約付きデコーディング（Grammar-Constrained Decoding）** が有効化され、Gemini のトークン生成自体がスキーマ文法に厳密に制約されるため、正規表現や文字列整形を一切使わずに `response.parsed` から検証済みの Pydantic インスタンスを直接取得できます。

![図4-1: Pydantic (response_schema) による制約付き構造化出力パイプライン](https://examples.drawlib.com/google-genai/images/04_structured_outputs/ch04_structured_pipeline_ja.png)

*図4-1: Pydantic (response_schema) による制約付き構造化出力パイプライン*

---

## 1. クライアント初期化とネストされた Pydantic スキーマ定義

まず、セルの先頭で `MODEL = "gemini-3.8-flash"` を定数として定義し（モデルの切り替えを1箇所で管理できるようにします）、Google Colab のシークレット（左サイドバーの `🔑` アイコン）に保存した `GEMINI_API_KEY` を読み込んで `google-genai` クライアントを初期化します。

In [ ]:
!pip install -q google-genai

In [ ]:
MODEL = "gemini-3.8-flash"

from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

次に、簡潔な障害報告文からポストモーテム情報を構造化抽出するため、ネストされた Pydantic スキーマ（`IncidentSummary` 内の `list[ActionItem]`）を定義します。`response_schema` に `IncidentSummary` クラスを渡すだけで、SDK が JSON Schema への変換と `response.parsed` へのデシリアライズを自動実行します（`pydantic` は Google Colab に標準搭載されており、`google-genai` の依存関係としても同梱されています）。

In [ ]:
from typing import Literal

from google.genai import types
from pydantic import BaseModel, Field


class ActionItem(BaseModel):
    owner: str = Field(description="対応タスクを担当するチーム名。")
    task: str = Field(description="具体的な再発防止施策。")
    priority: Literal["P0", "P1", "P2"] = Field(description="優先度（P0が最優先）。")


class IncidentSummary(BaseModel):
    title: str = Field(description="40文字以内の簡潔なインシデント件名。")
    severity: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"] = Field(
        description="顧客影響度に基づく重大度レベル。"
    )
    root_cause: str = Field(description="技術的な根本原因の1文要約。")
    affected_services: list[str] = Field(description="影響を受けたサービス名一覧。")
    action_items: list[ActionItem] = Field(description="優先度付きの再発防止タスク一覧。")


response = client.models.generate_content(
    model=MODEL,
    contents="Redisのタイムアウトにより、決済APIで12分間にわたり502エラーが発生しました。",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=IncidentSummary,
        temperature=0.1,
    ),
)

# response.parsed は IDE の型補完が効く検証済み IncidentSummary インスタンスを返す
summary: IncidentSummary = response.parsed
print(f"[{summary.severity}] {summary.title} (対策タスク: {len(summary.action_items)}件)")
print(summary.model_dump_json(indent=2))

---

## 2. スキーマ設計のベストプラクティスと高速・決定論的な構造化抽出

文法制約付きデコーディングは JSON の構文破損を100%防ぎますが、実務での **抽出精度** と **応答速度** をさらに高めるには、以下の4つの Pydantic スキーマ設計原則が有効です。

![図4-2: LLM 構造化抽出の精度を高める Pydantic スキーマ設計パターン](https://examples.drawlib.com/google-genai/images/04_structured_outputs/ch04_schema_best_practices_ja.png)

*図4-2: LLM 構造化抽出の精度を高める Pydantic スキーマ設計パターン*

`response_schema` 用のモデルを設計する際は、次の4点を意識してください。

1. **閉じたカテゴリには `Literal` または `Enum` を使う**: `Literal["P0", "P1", "P2"]` や `str, Enum` で選択肢を列挙することで、表記揺れや想定外ラベルによる後続処理の分岐ミスを防止できます。
2. **任意の `dict[str, Any]` ではなく `list[SubModel]` を合成する**: キー名が不定の辞書型は文法制約を効かせにくいため、明示的な属性を持つサブモデルの配列としてモデリングします。
3. **曖昧な指標には簡潔な `Field(description=...)` を添える**: `description` は JSON Schema にそのまま埋め込まれるため、プロンプト本文を肥大化させずに文字数制限や緊急度の判定基準をモデルへ伝達できます。
4. **高速・決定論的な抽出では `temperature=0.0`〜`0.2` と `thinking_level="low"` を組み合わせる**: `gemini-3.8-flash` で定型データ抽出や分類を行う場合、`thinking_config=types.ThinkingConfig(thinking_level="low")` を併用するとレイテンシと出力のばらつきを最小化できます。

これら4つの原則を組み合わせ、`Enum` と `list[RoutedIssue]` および `thinking_level="low"` を用いて、複数の課題を含む問い合わせ文を型安全な振り分けレコードへ分割し、`response.parsed` の各要素を直接ループ処理してみましょう。

In [ ]:
from enum import Enum


class TicketQueue(str, Enum):
    BILLING = "billing"
    AUTH = "auth"
    ANALYTICS = "analytics"


class RoutedIssue(BaseModel):
    queue: TicketQueue = Field(description="振り分け先のサポートキュー。")
    urgent: bool = Field(description="決済またはログイン不能な場合はTrue。")
    summary: str = Field(description="30文字以内の簡潔な課題要約。")


class TriageBatch(BaseModel):
    issues: list[RoutedIssue] = Field(description="メッセージから抽出された個別課題の一覧。")


triage_response = client.models.generate_content(
    model=MODEL,
    contents="請求書#8821で二重決済が発生しており、CSVレポートのエクスポートも失敗します。",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=TriageBatch,
        temperature=0.1,
        thinking_config=types.ThinkingConfig(thinking_level="low"),
    ),
)

batch: TriageBatch = triage_response.parsed
for item in batch.issues:
    flag = "URGENT" if item.urgent else "NORMAL"
    print(f"[{flag}] -> {item.queue.value}: {item.summary}")